In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
img_paths = [
    "/content/drive/MyDrive/content/img1.jpg",
    "/content/drive/MyDrive/content/img2.jpg",
    "/content/drive/MyDrive/content/img3.jpg",
    "/content/drive/MyDrive/content/img4.jpg"
]

Отобраны 3 open-source библиотеки: DeepFace, InsightFace, Face Recognition



DeepFace

Ссылка: GitHub: https://github.com/serengil/deepface

Возможности:
-	Поддержка 6 моделей: VGG-Face, Facenet, OpenFace, DeepFace, Dlib, ArcFace
-	Распознавание, демография, эмоции, возраст
-	Простое сравнение двух изображений

Преимущества:

-	Много моделей из коробки
-	Встроенная визуализация
-	Можно переключать backend

Требования:

- PyTorch / TensorFlow / MXNet — в зависимости от backend
- Поддержка CUDA (опционально)


Тестируем Deepface

In [ ]:
!pip install deepface

Извлекаем все лица с каждого изображения и сохраняем их в all_faces

In [ ]:
from deepface import DeepFace

img_paths = [
    "/content/drive/MyDrive/content/img1.jpg",
    "/content/drive/MyDrive/content/img2.jpg",
    "/content/drive/MyDrive/content/img3.jpg",
    "/content/drive/MyDrive/content/img4.jpg"
]

all_faces = []

for idx, path in enumerate(img_paths):
    faces = DeepFace.extract_faces(
        path,
        detector_backend="opencv",
        align=True,
        enforce_detection=False
    )
    for i, face_obj in enumerate(faces):
        all_faces.append({
            "img_index": idx + 1,
            "face_index": i + 1,
            "face": face_obj["face"]
        })

Показываем все найденные лица по одному с подписями

In [ ]:
import matplotlib.pyplot as plt

for f in all_faces:
    plt.imshow(f["face"])
    plt.title(f"img{f['img_index']} — лицо {f['face_index']}")
    plt.axis("off")
    plt.show()

Сохраняем каждое лицо как отдельный JPEG-файл во временную папку

In [ ]:
import os
import numpy as np
import cv2

temp_dir = "/content/deepface_faces"
os.makedirs(temp_dir, exist_ok=True)

for f in all_faces:
    img_arr = np.array(f["face"])
    if img_arr.dtype != np.uint8:
        img_arr = (np.clip(img_arr, 0, 1) * 255).astype(np.uint8)
    if img_arr.shape[2] != 3:
        img_arr = img_arr[:, :, :3]
    fname = f"img{f['img_index']}_face{f['face_index']}.jpg"
    f["temp_path"] = os.path.join(temp_dir, fname)
    cv2.imwrite(f["temp_path"], cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR))

Получаем эмбеддинги лиц через DeepFace с моделью ArcFace (векторные представления для сравнения), печатаем первые 5 значений эмбеддинга (для краткости)


In [ ]:
from deepface import DeepFace
import numpy as np

for f in all_faces:
    rep = DeepFace.represent(
        img_path=f["temp_path"],
        model_name="ArcFace",
        enforce_detection=False
    )
    f["embedding"] = rep[0]["embedding"]

    print(f"img{f['img_index']} лицо {f['face_index']} → embedding[:5] = {np.round(f['embedding'][:5], 4)} ...")

Определяем похожие пары и выводим на экран

Взято завышенное значение threshold = 4.5, чтобы выводить совпадения и не совпадения. Чтобы были только совпадения, параметр threshold нужно уменьшить до 3.0

Для каждой пары извлекаем эмбеддинги, считаем эвклидово расстояние и проверяем разницу.



In [ ]:
from numpy.linalg import norm
import matplotlib.pyplot as plt
import numpy as np

threshold = 4.5

similar_pairs = []
for i in range(len(all_faces)):
    for j in range(i + 1, len(all_faces)):
        e1 = np.array(all_faces[i]["embedding"])
        e2 = np.array(all_faces[j]["embedding"])
        dist = norm(e1 - e2)
        if dist < threshold:
            similar_pairs.append((i, j, dist))

for i, j, dist in sorted(similar_pairs, key=lambda x: x[2]):
    face1 = all_faces[i]["face"]
    face2 = all_faces[j]["face"]
    label1 = f"img{all_faces[i]['img_index']} лицо {all_faces[i]['face_index']}"
    label2 = f"img{all_faces[j]['img_index']} лицо {all_faces[j]['face_index']}"

    fig, axs = plt.subplots(1, 2, figsize=(5, 3))
    axs[0].imshow(face1)
    axs[0].axis("off")
    axs[0].set_title(label1, fontsize=8)

    axs[1].imshow(face2)
    axs[1].axis("off")
    axs[1].set_title(label2, fontsize=8)

    plt.suptitle(f"distance = {dist:.4f}", fontsize=10)
    plt.tight_layout()
    plt.show()

Тестирование библиотеки DeepFace успешно завершено

InsightFace

Ссылка: GitHub: https://github.com/deepinsight/insightface

Возможности:

-    Поддержка SOTA моделей: ArcFace, CurricularFace, CosFace и др
-    Быстрый inference с использованием MXNet, ONNX, PyTorch.
-    Простое сравнение двух изображений
-    Выделение эмбеддингов, выравнивание лиц; распознавание, верификация и кластеризация

Преимущества:

-    Высокая точность (используется в NIST benchmark)
-    Поддержка FP16 и ONNX Runtime для ускорения
-    Самая популярная библиотека в академической среде

Требования:

- CUDA для ускорения (желательно)
- ONNX Runtime или PyTorch
- Работает на CPU, но медленно

Тестируем Insightface

In [ ]:
!pip install -q insightface

In [ ]:
!pip install -q onnxruntime-gpu

Извлекаем все лица с каждого изображения и сохраняем их в all_faces

In [ ]:
from insightface.app import FaceAnalysis
import cv2

img_paths = [
    "/content/drive/MyDrive/content/img1.jpg",
    "/content/drive/MyDrive/content/img2.jpg",
    "/content/drive/MyDrive/content/img3.jpg",
    "/content/drive/MyDrive/content/img4.jpg"
]

app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

all_faces = []

for idx, path in enumerate(img_paths):
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    faces = app.get(img_rgb)

    for i, face in enumerate(faces):
        x1, y1, x2, y2 = map(int, face.bbox)
        face_crop = img_rgb[y1:y2, x1:x2]

        all_faces.append({
            "img_index": idx + 1,
            "face_index": i + 1,
            "face": face_crop,
            "face_obj": face
        })

Показываем все найденные лица по одному с подписями

In [ ]:
import matplotlib.pyplot as plt

for f in all_faces:
    plt.imshow(f["face"])
    plt.title(f"img{f['img_index']} — лицо {f['face_index']}")
    plt.axis("off")
    plt.show()

Сохраняем каждое лицо как отдельный JPEG-файл во временную папку

In [ ]:
import os
import numpy as np
import cv2

temp_dir = "/content/insightface_faces"
os.makedirs(temp_dir, exist_ok=True)

for f in all_faces:
    img_arr = np.array(f["face"])
    if img_arr.dtype != np.uint8:
        img_arr = (np.clip(img_arr, 0, 1) * 255).astype(np.uint8)
    if img_arr.shape[2] != 3:
        img_arr = img_arr[:, :, :3]

    fname = f"img{f['img_index']}_face{f['face_index']}.jpg"
    f["temp_path"] = os.path.join(temp_dir, fname)
    cv2.imwrite(f["temp_path"], cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR))

Получаем эмбеддинги лиц

In [ ]:
import numpy as np

for f in all_faces:
    embedding = f["face_obj"].embedding
    f["embedding"] = embedding
    print(f"img{f['img_index']} лицо {f['face_index']} → embedding[:5] = {np.round(embedding[:5], 4)} ...")

Определяем похожие пары и выводим на экран

Предварительно осуществляется нормализация эмбеддингов. Порог treshold завышен, чтобы были видны и совпадения, и не совпадения. При значении treshhod=1.0. будут только совпадения.

In [ ]:
from numpy.linalg import norm
import matplotlib.pyplot as plt
import numpy as np

for f in all_faces:
    emb = f["face_obj"].embedding
    f["embedding"] = emb / norm(emb)

similar_pairs = []
threshold = 1.35

for i in range(len(all_faces)):
    for j in range(i + 1, len(all_faces)):
        e1 = all_faces[i]["embedding"]
        e2 = all_faces[j]["embedding"]
        dist = norm(e1 - e2)
        if dist < threshold:
            similar_pairs.append((i, j, dist))

print(f"Найдено похожих пар: {len(similar_pairs)}")

for i, j, dist in sorted(similar_pairs, key=lambda x: x[2]):
    face1 = all_faces[i]["face"]
    face2 = all_faces[j]["face"]
    label1 = f"img{all_faces[i]['img_index']} лицо {all_faces[i]['face_index']}"
    label2 = f"img{all_faces[j]['img_index']} лицо {all_faces[j]['face_index']}"

    fig, axs = plt.subplots(1, 2, figsize=(5, 3))
    axs[0].imshow(face1)
    axs[0].axis("off")
    axs[0].set_title(label1, fontsize=8)

    axs[1].imshow(face2)
    axs[1].axis("off")
    axs[1].set_title(label2, fontsize=8)

    plt.suptitle(f"distance = {dist:.4f}", fontsize=10)
    plt.tight_layout()
    plt.show()

Тестирование библиотеки InsightFace успешно завершено

Face Recognition

Ссылка: GitHub: https://github.com/ageitgey/face_recognition

Возможности:

-    Простой API на базе dlib
-    Верификация и идентификация
-    Поддержка batch-инференса
-    Легко использовать для небольших задач и хобби-проектов

Преимущества:

-    Можно использовать даже без глубоких знаний ML
-    Прост в использовании

Требования:

- PyTorch / TensorFlow / MXNet — в зависимости от backend
- Поддержка CUDA (опционально)

Тестируем Face recognition

Тестирование показало, что библиотека face_recognition имеет существенные конфликты с Google Colab, связанные с её зависимостью от dlib.

Основные проблемы:

 - dlib не поддерживает GPU на Colab без ручной пересборки
 - ручная пересборка dlib под GPU или CPU нестабильна и часто приводит к ошибкам или зависанию
 - Даже при успешной сборке под CPU производительность будет низкой.


 Поэтому предпочтительнее использовать InsightFace или DeepFace, которые не зависят от dlib и корректно работают с GPU в Colab.

Выводы:

Библиотеки InsightFace и DeepFace продемонстрировали стабильную и корректную работу в среде Google Colab, обеспечивая высокую производительность, GPU-ускорение и схожие результаты распознавания лиц.

Библиотека Face Recognition показала низкую совместимость с Colab из-за нестабильной сборки и ограничений библиотеки dlib. Использование данной библиотеки не рекомендуется.